# Exploratory Data Analysis on the Marathos Dataset Part 1
- Athlete Year of Birth
- Athlete Average Speed
- Event distance/length
- Athlete Performance

## Loading the dataset

In [0]:
# Use the raw dataset inside the volume
VOLUME_PATH = "/Volumes/marathos_catalog/default/raw/"

# Use spark and sql
spark.sql(f"LIST '{VOLUME_PATH}'")

In [0]:
# See data path
DATA_PATH = "/Volumes/marathos_catalog/default/raw/data/"

display(spark.sql(f"LIST '{DATA_PATH}'"))

In [0]:
# Show what is inside the data folder. Using this method for better debugging later.
FILE_PATH = "/Volumes/marathos_catalog/default/raw/data/TWO_CENTURIES_OF_UM_RACES.csv"

# read csv
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

display(df)

## Schema Check

In [0]:
# Check the schema that Spark inferred from the CSV file | column name + data type
from pyspark.sql.functions import countDistinct, col, count, when, expr, lower, trim, regexp_replace, regexp_extract, lit
df.printSchema()

## Number of rows and columns

In [0]:
row_count = df.count()
column_count = len(df.columns)

print(f"Number of rows: {row_count}")
print(f"Number of columnss: {column_count}")


## Name of columns and number of distinct values

In [0]:
distinct_counts = []

for column in df.columns:
    distinct_count = df.agg(countDistinct(column).alias("distinct_values")).collect()[0]["distinct_values"]
    distinct_counts.append((column, distinct_count))

distinct_counts_df = spark.createDataFrame(
    distinct_counts,
    ["column_name", "distinct_values"]
)

display(distinct_counts_df)

## Descriptive Summary of numerical fields

The dataset has columns that contain numeric-looking values but still be stored as string.
I will divide these into different sections.

### Already numeric fields
```
Year of event                  integer
Event number of finishers      integer
Athlete year of birth          double
Athlete ID                     integer
```

###  Numeric but `string` type
```
Event distance/length -- investigate in EDA and clean more in silver
Athlete performance   -- investigate in EDA in Silver Layer later
Athlete average speed  -- cast to double ??
```

#### 1. Event distance/length
For basic EDA, inspect it as a categorical/text column first. Full cleaning can wait until Silver.
Need to extract things divided layter like this
```
distance_or_length_value = 50
distance_or_length_unit = km`
```

#### 2. Athlete average speed  -- cast to double


#### 3. Athelete Performance
```
4:51:39 h
5:15:45 h
```
Athlete performance contains duration values stored as strings, so I will not include it in the basic numeric summary until it is cleaned and converted. This is time duration, so it is not directly numeric yet. I will convert it later in Silver, probably into seconds, minutes, or hours.


## Descriptive summary for already numeric columns

In [0]:
# Descriptive summary of columns that Spark already inferred as numeric

already_numeric_columns = [
    "Year of event",
    "Event number of finishers",
    "Athlete year of birth"
]

display(df.select(already_numeric_columns).summary())

### Fimdings Descriptive summary of already numerical fields
I created a descriptive summary for the numerical columns that Spark inferred from the raw CSV file:
- `Year of event`
- `Event number of finishers`
- `Athlete year of birth`

- The dataset contains records from **1798 to 2022**, which aligns with the historical scope of the dataset. Most records are from more recent years, with the median event year being **2015**.

- For `Event number of finishers`, the median is **235**, while the mean is around **1452**. This shows that most events have a few hundred finishers, but some very large events increase the average. The maximum value is **20,027 finishers**.

- For `Athlete year of birth`, there are fewer non-null values compared to the other numerical columns, which means some athlete birth years are missing. The minimum value is **1193** and the maximum value is **2021**, so this column will need further validation during the Silver layer cleaning process.

- This summary helps identify the general distribution of the numerical fields and highlights potential data quality issues that should be handled later in the pipeline.

In [0]:
# Check anomaly in year values
# Check the lowest athlete year of birth values
lowest_birth_years_df = (
    df
    .groupBy("Athlete year of birth")
    .count()
    .orderBy(col("Athlete year of birth").asc())
)

display(lowest_birth_years_df)

Check: 
1. null values in Athlete year of birth
2. the clearly impossible value: 1193

In [0]:
# Inspect only null values and the clearly invalid birth year 1193
birth_year_issue_df = (
    df
    .filter(
        col("Athlete year of birth").isNull() |
        (col("Athlete year of birth") == 1193)
    )
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete club",
        "Athlete country",
        "Athlete year of birth",
        "Athlete gender",
        "Athlete age category",
        "Athlete average speed",
        "Athlete ID"
    )
    .orderBy(col("Athlete year of birth").asc_nulls_first())
)

display(birth_year_issue_df)

In [0]:
birth_year_quality_summary_df = df.select(
    count(when(col("Athlete year of birth").isNull(), True)).alias("null_birth_year_count"),
    count(when(col("Athlete year of birth") == 1193, True)).alias("invalid_1193_birth_year_count")
)

display(birth_year_quality_summary_df)

- The Athlete year of birth column contains 588,161 null values and one clearly invalid value: 1193.

- For the Silver layer, I will treat null birth years as missing data and the value 1193 as invalid. I will later calculate athlete_age_at_event to support cleaner age-based analysis.

## Athlete Average Speed
Add Athlete average speed by converting it to numeric for now because Athlete average speed is currently a string, create a temporary EDA version:

In [0]:
# Check malformed average speed values
athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

display(
    athlete_speed_df
    .filter(
        col("Athlete average speed").isNotNull()
        & col("athlete_average_speed_numeric").isNull()
    )
    .select("Athlete average speed")
    .distinct()
)

In [0]:
# Descriptive summary for Athlete average speed only
# Some values are malformed (sample above), so try_cast converts invalid values to NULL instead of failing.
athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

display(
    athlete_speed_df
    .select("athlete_average_speed_numeric")
    .summary()
)

### Findings descriptive summary of athlete average speed
- This column looked numerical, but Spark read it as a string. To analyze it, I temporarily converted it to a numeric column using `try_cast`.

- Most values seem reasonable. The median speed is **7.354**, and most values are between **5.783** and **9.1**.
- However, the maximum value is **29,644**, which is clearly too high for athlete average speed. This also makes the average much higher than expected.

- This tells me that the column has some incorrect or extreme values. I will handle these data quality issues later in the Silver layer.

#### Check the extremely high speeds

The values are not actually impossible speeds. They look like a decimal separator issue.
```
Event distance/length: 100km
Athlete performance: 3:32:00 h
Athlete average speed: 28302.0
```
- For a 100km race in about 3.53 hours, the speed should be around: 28.302 km/h
- So the values most likely means:
```
28302 → 28.302
29644 → 29.644
28942 → 28.942
```
- probably missing decimal formatting, not just random outliers.
- **Cleaning in Silver later: If athlete_average_speed_numeric > 1000, divide by 1000.**

In [0]:
# Convert Athlete average speed safely
athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

# Inspect raw athlete average speed values that look extremely high
speed_format_issue_df = (
    athlete_speed_df
    .filter(col("athlete_average_speed_numeric") > 1000)  # filter here
    .select(
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "athlete_average_speed_numeric",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
    .orderBy(col("athlete_average_speed_numeric").desc())
)

display(speed_format_issue_df)

In [0]:
# Convert Athlete average speed safely
athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

# Count rows where speed looks incorrectly scaled
high_speed_count = (
    athlete_speed_df
    .filter(col("athlete_average_speed_numeric") > 1000)
    .count()
)

print(f"Number of rows where athlete average speed is above 1000: {high_speed_count}")

In [0]:
# Convert Athlete average speed safely
athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

# Inspect speeds that are high, but not part of the >1000 scaling issue
medium_high_speed_df = (
    athlete_speed_df
    .filter(
        (col("athlete_average_speed_numeric") > 30) &
        (col("athlete_average_speed_numeric") <= 1000)
    )
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "athlete_average_speed_numeric",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
    .orderBy(col("athlete_average_speed_numeric").desc())
)

display(medium_high_speed_df)

- **Silver Layer later: Remove rows where Event distance/length ends with "d"**

In [0]:
# Check how many rows will be removed for d
# Count rows where event distance/length is day-based
day_based_event_count = (
    df
    .filter(lower(trim(col("Event distance/length"))).rlike(r"\d+d$"))
    .count()
)

print(f"Number of day-based event rows to remove later in Silver: {day_based_event_count}")

In [0]:
# Inspect day-based event rows
day_based_events_df = (
    df
    .filter(lower(trim(col("Event distance/length"))).rlike(r"\d+d$"))
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
    .orderBy("Event distance/length")
)

display(day_based_events_df)

- The dataset contains 12,134 day-based event rows, such as 10d races.

- These events store athlete performance as distance covered in km during a fixed number of days. For example, a 10d event has Athlete performance values such as 724.205 km or 1015.092 km.

- day-based events can be considered invalid for simplicity. Therefore, I will remove rows where Event distance/length contains day-based values such as 6d, 7d, 10d, or 18d in the Silver layer.

#### Check the valid event types to keep: km, mi, and h.
- The event distance/length column can be classified mostly into km, mi, and h-based events. 
- There are also 12,134 day-based rows, which will be removed in the Silver layer according to the project cleaning rule. 
- Additionally, 1,322 rows are classified as "other" and need further inspection before deciding the Silver cleaning strategy.

In [0]:
# Classify event distance/length into event unit type
event_unit_type_df = (
    df
    .withColumn(
        "event_unit_type",
        when(lower(trim(col("Event distance/length"))).rlike(r"\d+d$"), "days_invalid")
        .when(lower(trim(col("Event distance/length"))).contains("km"), "distance_km")
        .when(lower(trim(col("Event distance/length"))).contains("mi"), "distance_mi")
        .when(lower(trim(col("Event distance/length"))).rlike(r"\d+h$"), "time_hours")
        .otherwise("other")
    )
    .groupBy("event_unit_type")
    .count()
    .orderBy(col("count").desc())
)

display(event_unit_type_df)

#### What is that others?
**1. Missing event distance/length (None = 1053 rows)**
    - These should probably be removed later in Silver because we cannot classify the event type.

**2. Distance values with incomplete units**
    - These are probably kilometers, but written inconsistently.
```
 160k → 160km
50K  → 50km
105k → 105km
```

**3. Strange/malformed values**
    - REMOVE
```
07:35
6:40
71.5
45.1m´
```
    


In [0]:
# Inspect event distance/length values classified as "other"
other_event_distance_df = (
    df
    .withColumn(
        "event_unit_type",
        when(lower(trim(col("Event distance/length"))).rlike(r"\d+d$"), "days_invalid")
        .when(lower(trim(col("Event distance/length"))).contains("km"), "distance_km")
        .when(lower(trim(col("Event distance/length"))).contains("mi"), "distance_mi")
        .when(lower(trim(col("Event distance/length"))).rlike(r"\d+h$"), "time_hours")
        .otherwise("other")
    )
    .filter(col("event_unit_type") == "other")
    .groupBy("Event distance/length")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(other_event_distance_df)

#### inspect the k values

In [0]:
# Inspect Event distance/length values written with k or K instead of km
event_distance_k_format_df = (
    df
    .filter(lower(trim(col("Event distance/length"))).rlike(r"^[0-9]+(\.[0-9]+)?k$"))
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
    .orderBy("Event distance/length")
)

display(event_distance_k_format_df)

- Problem: Athlete average speed = 0.0
- That means the distance itself can probably be cleaned, but the speed column is not reliable for these rows.
- Some Event distance/length values use "k" or "K" instead of "km", for example 115k and 160k. 
- These appear to represent kilometer-based races and can be standardized in the Silver layer.

- However, all inspected rows have Athlete average speed equal to 0.0, even though Athlete performance contains valid-looking time durations. 
This means the event distance can be cleaned, but Athlete average speed = 0 may need to be removed

#### count how many k/K rows have speed 0

In [0]:
# Convert speed safely and check k/K event rows with zero speed
k_format_speed_zero_count = (
    df
    .withColumn(
        "athlete_average_speed_numeric",
        expr("try_cast(`Athlete average speed` as double)")
    )
    .filter(lower(trim(col("Event distance/length"))).rlike(r"^[0-9]+(\.[0-9]+)?k$"))
    .filter(col("athlete_average_speed_numeric") == 0)
    .count()
)

print(f"Number of k/K distance rows with athlete average speed = 0: {k_format_speed_zero_count}")

- There are 211 rows where Event distance/length is written with "k" or "K" instead of "km", for example 115k and 160k.

- All 211 of these rows have Athlete average speed equal to 0, even though Athlete performance contains valid-looking duration values such as 13:41:44 h or 35:43:23 h.

- This means the event distance format can be standardized in the Silver layer, but the Athlete average speed column is unreliable for these rows. In Silver, I will convert k/K values to km format and either recalculate average speed from distance and performance time or treat the original speed value as invalid.

#### inspect the remaining strange values
- 58 rows with unusual Event distance/length values.
- The "other" event distance/length values contain inconsistent or malformed values, such as None, 07:35, 6:40, 71.5, and 45.1m.
- To keep the Silver layer clean and reliable, I will remove these rows.

In the Silver layer, I will keep only clearly valid event distance/length units:
- km
- mi
- h

- I will remove day-based events ending in d and malformed/unknown values.

In [0]:
# Inspect remaining unusual event distance/length values
unusual_event_distance_df = (
    df
    .filter(
        col("Event distance/length").isin("45.1m", "07:35", "71.5", "6:40")
    )
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
    .orderBy("Event distance/length")
)

display(unusual_event_distance_df)

#### Check NONE

In [0]:
# Inspect rows where Event distance/length is missing or "None"
missing_event_distance_df = (
    df
    .filter(
        col("Event distance/length").isNull() |
        (col("Event distance/length") == "None")
    )
    .select(
        "Year of event",
        "Event dates",
        "Event name",
        "Event distance/length",
        "Athlete performance",
        "Athlete average speed",
        "Athlete country",
        "Athlete gender",
        "Athlete ID"
    )
)

display(missing_event_distance_df)

- Rows where Event distance/length is None have unreliable event information.
- The event type cannot be classified, and Athlete performance contains malformed values such as time strings ending with km.
- These rows should be removed in the Silver layer because they cannot support valid distance, time, or speed analysis.

## Event distance/length
I sse that this column is a mess. 
So the best EDA approach is:

- Count distinct values.
- Show the most common values.
- Extract the numeric part and unit temporarily for analysis.
- Count how many records are km, mi, h, etc.
- Later in Silver, create proper cleaned columns.

In [0]:
# Count distinct values in Event distance/length

distinct_distance_count = df.select("Event distance/length").distinct().count()

print(f"Number of distinct Event distance/length values: {distinct_distance_count}")

In [0]:
# Show most common Event distance/length values
# Which race types appear most often in the dataset?
event_distance_counts_df = (
    df.groupBy("Event distance/length")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_distance_counts_df)

In [0]:
# Inspect multi-stage event distance/length values
# SHow only Etappen

multistage_events_df = (
    df
    .filter(col("Event distance/length").contains("Etappen"))
    .groupBy("Event distance/length")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(multistage_events_df)

In [0]:
# Count multi-stage rows

total_rows = df.count()

multistage_row_count = (
    df
    .filter(col("Event distance/length").contains("Etappen"))
    .count()
)

print(f"Total rows: {total_rows}")
print(f"Multi-stage rows: {multistage_row_count}")
print(f"Multi-stage percentage: {(multistage_row_count / total_rows) * 100:.2f}%")

### Count Athlete Event distance/length 

In [0]:
# Better temporary parsing of Event distance/length for EDA

event_distance_eda_df = (
    df
    .withColumn(
        "distance_or_length_value_raw",
        regexp_extract(col("Event distance/length"), r"^([0-9]+(?:\.[0-9]+)?)", 1)
    )
    .withColumn(
        "distance_or_length_value",
        expr("try_cast(distance_or_length_value_raw as double)")
    )
    .withColumn(
        "distance_or_length_unit",
        regexp_extract(col("Event distance/length"), r"^[0-9]+(?:\.[0-9]+)?([a-zA-Z]+)", 1)
    )
    .withColumn(
        "stage_count_raw",
        regexp_extract(col("Event distance/length"), r"/([0-9]+)Etappen", 1)
    )
    .withColumn(
        "stage_count",
        when(
            col("stage_count_raw") != "",
            expr("try_cast(stage_count_raw as int)")
        ).otherwise(lit(1))
    )
    .withColumn(
        "event_type",
        when(col("Event distance/length").contains("Etappen"), "distance_multistage")
        .when(col("distance_or_length_unit").isin("km", "mi"), "distance")
        .when(col("distance_or_length_unit") == "h", "time")
        .otherwise("unknown")
    )
)

display(
    event_distance_eda_df
    .select(
        "Event distance/length",
        "distance_or_length_value",
        "distance_or_length_unit",
        "stage_count",
        "event_type"
    )
    .distinct()
    .orderBy("event_type", "distance_or_length_unit", "distance_or_length_value")
)

In [0]:

# Count event types based on temporary EDA parsing

event_type_counts_df = (
    event_distance_eda_df
    .groupBy("event_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_type_counts_df)

### Findings in `Event distance/length`
- The `Event distance/length` column contains different race formats and is stored as a string in the raw dataset.
- **2,160 distinct values** in this column. The most common values include standard distance-based races such as `50km`, `100km`, `50mi`, and `100mi`, as well as time-based races such as `6h`, `12h`, and `24h`.

- I also found values such as:
```text
57km/2Etappen
 60mi/2Etappen
 68km/3Etappen
250km/6Etappen
```
- These values appear to represent multi-stage races. The word `Etappen` likely means stages or legs, so a value such as `57km/2Etappen` can be interpreted as 57 kilometers over 2 stages.

#### Event types:
- `distance`: standard distance-based events
- `time`: time-based events
- `distance_multistage`: distance-based events completed over multiple stages
- `unknown`: values that need further investigation
- **In the Silver layer, this unknown column should be cleaned into separate structured columns so that the data can be validated, modeled, and used in the Gold layer.**

#### Checking unknown event type

In [0]:
# Inspect unknown Event distance/length values

unknown_event_distance_df = (
    event_distance_eda_df
    .filter(col("event_type") == "unknown")
    .groupBy("Event distance/length")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(unknown_event_distance_df)

#### Cleaning in event distance/length later
1. Convert values to lowercase
2. Remove extra spaces
3. Extract number and unit
4. Classify km/mi as distance
5. Classify h as time
6. Classify Etappen as distance_multistage
7. Remove d/day-based rows as invalid
8. Remove or flag None/null rows

## Athlete performance

In [0]:
# Inspect Athlete performance values
athlete_performance_counts_df = (
    df
    .groupBy("Athlete performance")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_performance_counts_df)

- Athlete performance is not always time. I also have distance values.
- **So later in Silver, Ineed to validate that if the event is km or mi, performance should be in h; if the event is h, performance should be in km.**



### Count performance units
The Athlete performance column contains mixed formats. Most values are time-based, but some values are distance-based. This is expected because time-based events measure athlete performance by distance completed, while distance-based events measure performance by finishing time.

In [0]:
# Count performance value types/units

athlete_performance_type_df = (
    df
    .withColumn(
        "performance_type",
        when(col("Athlete performance").isNull(), "null")
        .when(col("Athlete performance").endswith(" h"), "time_hours")
        .when(col("Athlete performance").endswith(" km"), "distance_km")
        .when(col("Athlete performance").endswith(" mi"), "distance_mi")
        .otherwise("unknown")
    )
    .groupBy("performance_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_performance_type_df)

- Most rows are stored as time-based performance values, such as `8:52:00 h`. These rows likely belong to distance-based events, where the athlete performance is measured by finishing time.

- Some rows are stored as distance-based performance values, such as `160.944 km`. These rows likely belong to time-based events, where the athlete performance is measured by distance completed within a fixed time limit.

- The result also shows that there are only **2 null values** in this column.

- This EDA step is important because the Silver layer will need to validate the relationship between `Event distance/length` and `Athlete performance`:
- If the event unit is `km` or `mi`, the athlete performance should be measured in **hours.**
- If the event unit is `h`, the athlete performance should be measured in **kilometers.**


In [0]:
# Inspect rows where Athlete performance is null

display(
    df
    .filter(col("Athlete performance").isNull())
)

There are only 2 rows where Athlete performance is null, and these rows also have missing athlete-level information. These records should likely be removed later in the Silver cleaning layer because they are incomplete result records.
